# Spark Setup

In [1]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

Debug: SparkSession has been created successfully.


Accepting streams

In [2]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1")
camera_stream_b = read_camera_stream("camera-events-B", "2")
camera_stream_c = read_camera_stream("camera-events-C", "3")

print("Debug: Kafka streams have been created for all three cameras.")

Debug: Kafka streams have been created for all three cameras.


join them with each other so easy process i guess idk ill figure out why later

In [3]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

+---------+--------+-----------+
|camera_id|position|speed_limit|
+---------+--------+-----------+
|        1|   152.5|        110|
|        2|   153.5|        110|
|        3|   154.5|         90|
+---------+--------+-----------+

Debug: Camera loaded: 3 cameras.


In [4]:
# def get_instant_violations(stream):
#     return (
#         stream
#         .filter(col("speed_reading") > col("speed_limit"))
#         .withColumn("violation_type", lit("instantaneous"))
#         .withColumn("violation_date", to_date(col("event_time")))
#         .select(
#             "car_plate",
#             "violation_date",
#             "violation_type",
#             "camera_id",
#             col("speed_reading").alias("speed_recorded"),
#             "speed_limit",
#             col("event_time").cast("string").alias("event_time"),
#             "source" 
#         )
#     )

# camera_a_instant_violations = get_instant_violations(joined_stream_a)
# camera_b_instant_violations = get_instant_violations(joined_stream_b)
# camera_c_instant_violations = get_instant_violations(joined_stream_c)

# # Combine all instant violations into one stream
# all_instant_violations = (
#     camera_a_instant_violations
#     .union(camera_b_instant_violations)
#     .union(camera_c_instant_violations)
# )

# print("Debug: Instantaneous violations have been extracted and combined.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger


average speed violations time baby

In [ ]:
stream_a_query = (
    camera_stream_a
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch("Camera Stream A"))
    .start()
)

stream_b_query = (
    camera_stream_b
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch("Camera Stream B"))
    .start()
)

# A→B segment join (camera 1 to camera 2)
segment_ab = (
    camera_stream_a.alias("entry")
    .join(
        camera_stream_b.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.speed_reading").alias("entry_speed"),
        col("exit.speed_reading").alias("exit_speed"),
    )
)

# # B→C segment join (camera 2 to camera 3)
# segment_bc = (
#     joined_stream_b.alias("entry")
#     .join(
#         joined_stream_c.alias("exit"),
#         expr("""
#             entry.car_plate = exit.car_plate
#             AND exit.event_time > entry.event_time
#             AND exit.event_time <= entry.event_time + interval 10 minutes
#         """),
#         "inner"
#     )
#     .select(
#         col("entry.car_plate").alias("car_plate"),
#         col("entry.camera_id").alias("start_camera_id"),
#         col("exit.camera_id").alias("end_camera_id"),
#         col("entry.event_time").alias("entry_time"),
#         col("exit.event_time").alias("exit_time"),
#         col("entry.position").alias("entry_position"),
#         col("exit.position").alias("exit_position"),
#         col("exit.speed_limit").alias("speed_limit"),
#         col("exit.source").alias("source")
#     )
# )

segment_ab = (
    segment_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch("Segment A→B"))
    .start()
)

# segment_bc = (
#     segment_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Segment B→C"))
#     .start()
# )

spark.streams.awaitAnyTermination()


[2026-05-12 06:52:16] Camera Stream B
Spark Batch ID: 0
Rows received: 0

[2026-05-12 06:52:16] Camera Stream A
Spark Batch ID: 0
Rows received: 0
+--------+--------+---------+---------+---------+-------------+----------+------+
|event_id|batch_id|car_plate|camera_id|timestamp|speed_reading|event_time|source|
+--------+--------+---------+---------+---------+-------------+----------+------+
+--------+--------+---------+---------+---------+-------------+----------+------+

+--------+--------+---------+---------+---------+-------------+----------+------+
|event_id|batch_id|car_plate|camera_id|timestamp|speed_reading|event_time|source|
+--------+--------+---------+---------+---------+-------------+----------+------+
+--------+--------+---------+---------+---------+-------------+----------+------+


[2026-05-12 06:52:20] Segment A→B
Spark Batch ID: 0
Rows received: 0
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plat


[2026-05-12 06:52:27] Segment A→B
Spark Batch ID: 3
Rows received: 0
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+


[2026-05-12 06:52:28] Camera Stream B
Spark Batch ID: 2
Rows received: 1
+------------------------------------+--------+---------+---------+--------------------------+-------------+--------------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp                 |speed_reading|event_time                |source|
+------------------------------------+--------+---------+---------+--------------------------+-------------+-

+---------+---------------+-------------+--------------+-------------+-------------------+--------------------------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time         |exit_time                 |entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+-------------------+--------------------------+-----------+----------+
|WC 89    |1              |2            |1             |3            |2024-01-01 08:00:03|2024-01-01 08:00:41.774758|96.7       |93.5      |
+---------+---------------+-------------+--------------+-------------+-------------------+--------------------------+-----------+----------+


[2026-05-12 06:52:35] Segment A→B
Spark Batch ID: 8
Rows received: 0
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_s


[2026-05-12 06:52:40] Segment A→B
Spark Batch ID: 11
Rows received: 0
+------------------------------------+--------+---------+---------+-------------------+-------------+-------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp          |speed_reading|event_time         |source|
+------------------------------------+--------+---------+---------+-------------------+-------------+-------------------+------+
|1dd988c0-d000-489b-8cf0-2cd738fb0565|4       |BQN 88   |1        |2024-01-01T08:19:48|61.3         |2024-01-01 08:19:48|1     |
|9478323b-a2f6-4592-8c9f-a9de5bb984ab|4       |ZEA 3530 |1        |2024-01-01T08:19:48|153.3        |2024-01-01 08:19:48|1     |
|3e613964-b717-44a8-bf81-a1290ae07a35|4       |SJ 15    |1        |2024-01-01T08:19:48|146.2        |2024-01-01 08:19:48|1     |
|9f538681-69f0-4efd-a3e3-ef44ee2c1094|4       |PB 55    |1        |2024-01-01T08:19:49|111.2        |2024-01-01 08:19:49|1     |
|26e1dd39-29ec-4692-bca0-5

+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+


[2026-05-12 06:52:48] Camera Stream B
Spark Batch ID: 6
Rows received: 2

[2026-05-12 06:52:48] Segment A→B
Spark Batch ID: 14
Rows received: 2
+------------------------------------+--------+---------+---------+--------------------------+-------------+--------------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp                 |speed_reading|event_time                |source|
+------------------------------------+--------+---------+---------+--------------------------+-------------+

...

+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+


[2026-05-12 06:53:08] Camera Stream B
Spark Batch ID: 10
Rows received: 2

[2026-05-12 06:53:08] Segment A→B
Spark Batch ID: 26
Rows received: 2
+------------------------------------+--------+---------+---------+--------------------------+-------------+--------------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp                 |speed_reading|event_time                |source|
+------------------------------------+--------+---------+---------+--------------------------+-------------

+---------+---------------+-------------+--------------+-------------+-------------------+--------------------------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time         |exit_time                 |entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+-------------------+--------------------------+-----------+----------+
|YRV 3    |1              |2            |2             |11           |2024-01-01 08:08:05|2024-01-01 08:08:58.749711|65.8       |63.8      |
+---------+---------------+-------------+--------------+-------------+-------------------+--------------------------+-----------+----------+


[2026-05-12 06:53:15] Segment A→B
Spark Batch ID: 30
Rows received: 0
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_

+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+


[2026-05-12 06:53:23] Camera Stream B
Spark Batch ID: 13
Rows received: 1

[2026-05-12 06:53:23] Segment A→B
Spark Batch ID: 34
Rows received: 1
+------------------------------------+--------+---------+---------+--------------------------+-------------+--------------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp                 |speed_reading|event_time                |source|
+------------------------------------+--------+---------+---------+--------------------------+-------------


[2026-05-12 06:53:31] Segment A→B
Spark Batch ID: 37
Rows received: 0
+------------------------------------+--------+---------+---------+-------------------+-------------+-------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp          |speed_reading|event_time         |source|
+------------------------------------+--------+---------+---------+-------------------+-------------+-------------------+------+
|febf3a02-f2b1-47e5-86a0-0ff64f40f886|14      |ID 3     |1        |2024-01-01T09:38:25|115.6        |2024-01-01 09:38:25|1     |
|2e07dc96-f030-495f-b822-1b21f8d0238c|14      |TJZ 83   |1        |2024-01-01T09:38:29|139.5        |2024-01-01 09:38:29|1     |
|82c2b30f-ae78-44cc-85ae-659c3d9d44a0|14      |XQS 7    |1        |2024-01-01T09:38:26|102.9        |2024-01-01 09:38:26|1     |
|64fcc375-d9da-4ad4-8ea3-14db42ff1653|14      |WPP 30   |1        |2024-01-01T09:38:26|123.6        |2024-01-01 09:38:26|1     |
|8afb1549-5e7b-4f2a-9cda-1

+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
|car_plate|start_camera_id|end_camera_id|entry_batch_id|exit_batch_id|entry_time|exit_time|entry_speed|exit_speed|
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+
+---------+---------------+-------------+--------------+-------------+----------+---------+-----------+----------+


[2026-05-12 06:53:38] Camera Stream B
Spark Batch ID: 16
Rows received: 2

[2026-05-12 06:53:38] Segment A→B
Spark Batch ID: 40
Rows received: 2
+------------------------------------+--------+---------+---------+--------------------------+-------------+--------------------------+------+
|event_id                            |batch_id|car_plate|camera_id|timestamp                 |speed_reading|event_time                |source|
+------------------------------------+--------+---------+---------+--------------------------+-------------